# 03 — SRA Reproduction

Reproduce the Subspace Reliability Analysis pipeline:
1. Extract features from the dataset
2. Fit the SRA model
3. Compute scores and evaluate (EER, ROC, DET)
4. Compare with baseline methods

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, det_curve

from radar_drone_vision.datasets.synthetic import SyntheticGenerator
from radar_drone_vision.signal.framing import frame_signal
from radar_drone_vision.signal.complex_log_fft import regularized_complex_log_fft
from radar_drone_vision.classical.sra import SubspaceReliabilityAnalysis
from radar_drone_vision.eval.metrics import compute_all_metrics
from radar_drone_vision.eval.eer import compute_eer, compute_far_at_frr

## 1. Generate synthetic dataset and extract features

In [ ]:
gen = SyntheticGenerator(seed=42, sample_duration_s=0.25, sample_rate_hz=2000.0)
samples = gen.generate_balanced_dataset(n_per_class=200)

FRAME_SIZE = 64
HOP_SIZE = 32
N_FFT = 64

features, labels = [], []
for s in samples:
    frames = frame_signal(s.signal, frame_size=FRAME_SIZE, hop_size=HOP_SIZE)
    feat = regularized_complex_log_fft(frames, n_fft=N_FFT, feature_mode="real_imag_concat")
    features.append(feat.ravel())
    labels.append(s.label_binary)

X = np.array(features)
y = np.array(labels)
print(f"Feature matrix: {X.shape}, Labels: {y.shape}, UAV: {y.sum()}, non-UAV: {(1-y).sum()}")

## 2. Train/test split and fit SRA

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=42
)

sra = SubspaceReliabilityAnalysis(m_uav=5, m_non_uav=20, ridge=1e-4)
sra.fit(X_train, y_train)
print("SRA fitted.")
print(f"  UAV subspace:     {sra.phi1_.shape}")
print(f"  non-UAV subspace: {sra.phi2_.shape}")

## 3. Compute scores, EER, FAR@FRR=1%

In [ ]:
ratios = sra.score_ratio(X_test)
# Invert scores so that higher = more UAV-like (for ROC/EER convention)
scores = -ratios

eer, eer_thresh = compute_eer(y_test, scores)
far_at_1 = compute_far_at_frr(y_test, scores, target_frr=0.01)

y_pred = sra.predict(X_test, threshold=1.0)
metrics = compute_all_metrics(y_test, y_pred, y_scores=scores)

print(f"EER:          {eer:.4f}")
print(f"EER threshold:{eer_thresh:.4f}")
print(f"FAR@FRR=1%:   {far_at_1:.4f}")
print(f"Accuracy:     {metrics['accuracy']:.4f}")
print(f"F1:           {metrics['f1']:.4f}")
print(f"AUC:          {metrics['auc']}")

## 4. Plot ROC and DET curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, scores)
fpr_det, fnr_det = det_curve(y_test, scores)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, lw=2)
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set_xlabel("False Positive Rate (FAR)")
axes[0].set_ylabel("True Positive Rate (1 - FRR)")
axes[0].set_title(f"ROC Curve (AUC={metrics['auc']:.3f})")
axes[0].grid(True, alpha=0.3)

axes[1].plot(fpr_det, fnr_det, lw=2, color="#e74c3c")
axes[1].set_xlabel("False Alarm Rate (FAR)")
axes[1].set_ylabel("Miss Rate (FRR)")
axes[1].set_title(f"DET Curve (EER={eer:.3f})")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Comparison table placeholder

| Method | Feature | Classifier | EER | FAR@FRR=1% | Notes |
|--------|---------|------------|-----|------------|-------|
| Paper (Thales X-band) | Proposed | SRA | 0.034 | — | Reference target |
| This notebook (synthetic) | Proposed | SRA | *see above* | *see above* | Synthetic data only |